## Ch11. Hierarchical and grouped time series Forecasting: Principles & Practice (Python Edition) Extracted from: fpppy-11-hierarchical-forecasting.qmd

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## [Slide 3] Australian Tourism Example

In [2]:
import pandas as pd
from hierarchicalforecast.utils import aggregate

spec = [
    ["Country"],
    ["Country", "State"],
    ["Country", "State", "Region"],
]
Y_df, S_df, tags = aggregate(
    df=aus_tourism.drop(columns=["unique_id"]),
    spec=spec
)

NameError: name 'aus_tourism' is not defined

## [Slide 6] Australian Prison Population

In [3]:
prison = pd.read_csv("data/prison.csv", index_col=0).assign(
    ds=lambda x: pd.PeriodIndex(x["ds"], freq="Q").to_timestamp(),
    Country="Australia",
)

spec = [
    ["Country"],
    ["Country", "State"],
    ["Country", "Gender"],
    ["Country", "Legal"],
    ["Country", "State", "Gender"],
    ["Country", "State", "Legal"],
    ["Country", "Legal", "Gender"],
    ["Country", "State", "Legal", "Gender"],
]
Y_df, S_df, tags = aggregate(df=prison, spec=spec)

## [Slide 8] Mixed Structure: Tourism with Purpose

In [4]:
spec = [
    ["Country", "State"],
    ["Country", "Purpose"],
    ["Country", "State", "Purpose"],
    ["Country", "State", "Region", "Purpose"],
]
Y_df, S_df, tags = aggregate(
    df=aus_tourism.drop(columns=["unique_id"]),
    spec=spec,
)

NameError: name 'aus_tourism' is not defined

## [Slide 11] Bottom-Up: Code

In [5]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

reconcilers = [BottomUp()]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)

sf = StatsForecast(
    models=[AutoETS(season_length=4)],
    freq="Q", n_jobs=-1
)
Y_hat_df = sf.forecast(h=4, df=Y_df)

Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df,
    Y_df=Y_df,
    S_df=S_df,
    tags=tags
)

## [Slide 14] Top-Down: Code

In [6]:
from hierarchicalforecast.methods import TopDown

Method 1: average historical proportions

In [7]:
reconcilers = [TopDown(method="average_proportions")]

Method 2: proportions of historical averages

In [8]:
reconcilers = [TopDown(method="proportion_averages")]

Method 3: forecast proportions (recommended)

In [9]:
reconcilers = [TopDown(method="forecast_proportions")]

hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df,
    Y_df=Y_df,
    S_df=S_df,
    tags=tags
)

ValueError: The reconciliation method(s) 'TopDown_method-forecast_proportions' require a strictly hierarchical structure. The provided hierarchy contains nodes with multiple parents (grouped structure), which is not supported by these methods. Please use a different reconciliation method (e.g., BottomUp, MinTrace, or ERM) that supports grouped hierarchies.

## [Slide 16] Middle-Out Approach

In [10]:
from hierarchicalforecast.methods import MiddleOut

reconcilers = [
    MiddleOut(
        middle_level=2,
        top_down_method="forecast_proportions"
    )
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_df,
    S_df=S_df, tags=tags
)

ValueError: The reconciliation method(s) 'MiddleOut_middle_level-2_top_down_method-forecast_proportions' require a strictly hierarchical structure. The provided hierarchy contains nodes with multiple parents (grouped structure), which is not supported by these methods. Please use a different reconciliation method (e.g., BottomUp, MinTrace, or ERM) that supports grouped hierarchies.

## [Slide 23] MinT Code

In [11]:
from hierarchicalforecast.methods import MinTrace

reconcilers = [
    BottomUp(),
    MinTrace(method="ols"),
    MinTrace(method="wls_var"),
    MinTrace(method="wls_struct"),
    MinTrace(method="mint_shrink"),  # recommended
]

hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df,
    Y_df=Y_fitted_df,   # needs fitted values for W estimation
    S_df=S_df,
    tags=tags
)

NameError: name 'Y_fitted_df' is not defined

## [Slide 25] Full Workflow

In [12]:
from statsforecast.models import AutoETS

spec = [
    ["Country"],
    ["Country", "State"],
    ["Country", "Purpose"],
    ["Country", "State", "Region"],
    ["Country", "State", "Purpose"],
    ["Country", "State", "Region", "Purpose"],
]
Y_df, S_df, tags = aggregate(
    aus_tourism.drop(columns=["unique_id"]), spec)

Y_train_df = Y_df.loc[lambda x: x["ds"] < "2016"]
Y_test_df  = Y_df.loc[lambda x: x["ds"] >= "2016"]

sf = StatsForecast(
    models=[AutoETS(season_length=4)],
    freq="Q", n_jobs=-1
)
Y_hat_df    = sf.forecast(h=8, df=Y_train_df, fitted=True)
Y_fitted_df = sf.forecast_fitted_values()

NameError: name 'aus_tourism' is not defined

## [Slide 27] Reconciliation and Evaluation

In [13]:
from hierarchicalforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mase
from functools import partial

reconcilers = [
    BottomUp(),
    MinTrace(method="ols"),
    MinTrace(method="mint_shrink"),
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_fitted_df,
    S_df=S_df, tags=tags
)

eval_tags = {
    "Total":   tags["Country"],
    "Purpose": tags["Country/Purpose"],
    "State":   tags["Country/State"],
    "Regions": tags["Country/State/Region"],
    "Bottom":  tags["Country/State/Region/Purpose"],
}
eval_df = Y_rec_df.merge(Y_test_df, on=["unique_id", "ds"])
evaluation = evaluate(
    df=eval_df, tags=eval_tags, train_df=Y_train_df,
    metrics=[rmse, partial(mase, seasonality=4)],
)

NameError: name 'Y_fitted_df' is not defined

## [Slide 29] Coherent Probabilistic Forecasts

In [14]:
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_fitted_df,
    S_df=S_df, tags=tags,
    intervals_method='normality'
)

NameError: name 'Y_fitted_df' is not defined

## [Slide 31] Bootstrap Approach

In [15]:
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_fitted_df,
    S_df=S_df, tags=tags,
    intervals_method='bootstrap'
)

from utilsforecast.losses import scaled_crps, mqloss

evaluation = evaluate(
    df=eval_df, tags=eval_tags, train_df=Y_train_df,
    metrics=[partial(mase, seasonality=4), mqloss],
    level=list(range(10, 100, 10)),
)

NameError: name 'Y_fitted_df' is not defined

## [Slide 33] Grouped Structure Application

In [16]:
from statsforecast.models import Naive

spec = [
    ["Country"],
    ["Country", "State"],
    ["Country", "Gender"],
    ["Country", "Legal"],
    ["Country", "State", "Gender", "Legal"],
]
prison_scaled = prison.assign(y=prison["y"] / 1e3)
Y_df, S_df, tags = aggregate(prison_scaled, spec)

Y_train_df = Y_df.loc[lambda x: x["ds"] < "2015"]
Y_test_df  = Y_df.loc[lambda x: x["ds"] >= "2015"]

models = [AutoETS(season_length=4, model="MAM"), Naive()]
sf = StatsForecast(models=models, freq="QS", n_jobs=-1)

## [Slide 35] Prison Population: Reconciliation

In [17]:
levels = list(range(10, 100, 10))
sf.fit(df=Y_train_df)
Y_hat_df    = sf.forecast(h=8, df=Y_train_df,
                           fitted=True, level=levels)
Y_fitted_df = sf.forecast_fitted_values()

reconcilers = [
    BottomUp(),
    MinTrace(method="mint_shrink"),
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_fitted_df,
    S_df=S_df, tags=tags, level=levels
)